<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/11-catalyst-optimizer/04-adaptive-query-execution.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 11.4 — Adaptive Query Execution: the optimizer that runs twice

AQE is **on** here (the one notebook that un-pins it). Watch it
coalesce 200 partitions down to a handful, switch a sort-merge join to
a broadcast at runtime, and see why `explain()` and the final plan
disagree.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("11.4-aqe")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "200")   # deliberately high — let AQE coalesce
    .config("spark.sql.adaptive.enabled", "true")    # AQE ON — this is the topic
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## Rewrite 1 — coalescing: 200 partitions → the few you needed

A groupBy with 50 keys produces ~50 non-empty partitions, not 200.
AQE merges the small ones after the shuffle. `explain()` shows the
*initial* plan; the executed plan shows `AQEShuffleRead coalesced`.

In [ ]:
df = spark.range(2_000_000).withColumn("k", (col("id") % 50).cast("int")) \
                             .withColumn("v", col("id") * 2)
agg = df.groupBy("k").sum("v")

print("INITIAL plan (explain — before execution):")
agg.explain()

agg.collect()   # run it, so AQE can re-plan with real sizes
print("\nFINAL plan (after AQE):")
print(agg.queryExecution.executedPlan)   # look for AdaptiveSparkPlan / AQEShuffleRead coalesced

## The 10.3 rule, demonstrated: AQE only merges, never splits

Confirm the output partition count collapsed far below 200. This is
why you set `shuffle.partitions` high and let AQE size down — a high
value is cheap, a low one AQE cannot fix.

In [ ]:
print("output partitions after AQE:",
      agg.rdd.getNumPartitions())   # << 200, coalesced to what the data needed

## Rewrite 2 — join strategy switch at runtime

Force the static planner to *not* broadcast (threshold tricks it), but
let AQE discover the real (tiny) size after the first shuffle and flip
SortMergeJoin → BroadcastHashJoin.

In [ ]:
# a filter whose selectivity the static planner can't know shrinks the right side
big   = spark.range(5_000_000).withColumn("k", (col("id") % 1000).cast("int"))
small = (spark.range(5_000_000).withColumn("k", (col("id") % 1000).cast("int"))
         .filter(col("id") < 20))     # actually tiny at runtime, huge on paper

joined = big.join(small, "k")

print("INITIAL plan — likely SortMergeJoin (two Exchanges):")
joined.explain()

joined.collect()
print("\nFINAL plan — AQE may have switched to broadcast:")
print(joined.queryExecution.executedPlan)   # look for BroadcastHashJoin + 'changed by AQE'-style nodes

## Why explain() lied: the SQL tab is the truth

`explain()` prints the plan *before* AQE. The Spark UI's SQL tab
(`spark_ui(spark)`) shows the final adaptive plan with runtime metrics.
Programmatically, `executedPlan` after a run gives you the final one.

In [ ]:
print("Initial (explain) vs final (executedPlan) — count the Exchanges in each:")
print("initial Exchanges:", joined.queryExecution.sparkPlan.toString().count("Exchange"))
print("final   Exchanges:", joined.queryExecution.executedPlan.toString().count("Exchange"))
# AQE's broadcast switch removes shuffles that explain() still showed

## The knobs (and the honest default: leave them on)

The relevant switches, printed. In production you keep these defaults;
AQE fixes the common mistakes cheaply. What it does NOT fix: unneeded
shuffles, skewed *aggregations* (join-only — Module 10.4), or a UDF
that killed codegen (11.3).

In [ ]:
for k in ["spark.sql.adaptive.enabled",
          "spark.sql.adaptive.coalescePartitions.enabled",
          "spark.sql.adaptive.skewJoin.enabled",
          "spark.sql.adaptive.advisoryPartitionSizeInBytes"]:
    print(f"{k} = {spark.conf.get(k)}")

In [ ]:
spark.stop()

## Exercises

1. Re-run the coalescing demo with AQE **off**
   (`spark.sql.adaptive.enabled=false`) and check
   `agg.rdd.getNumPartitions()`. Why is it exactly 200 now? Connect
   this to the 10.3 asymmetry.
2. Lower `advisoryPartitionSizeInBytes` to `1m` and re-run. Do you get
   *more* coalesced partitions? Why does a smaller target mean less
   merging?
3. Build a genuinely skewed join (one key with 90% of rows, like
   10.4) with AQE on. Find `AQEShuffleRead ... skewed` in the executed
   plan. Now the trap: build a skewed **groupBy** instead — does AQE
   skew handling help? Why not?
4. For the join-switch demo, print both `sparkPlan` (pre-AQE) and
   `executedPlan` (post-AQE) in full. Point to the exact node that
   changed and name the rewrite.
5. Turn AQE off and re-time the whole notebook's join. Is it slower?
   Where does the extra time go (hint: the second Exchange + sort AQE
   cancelled)?

In [ ]:
# your exercise code here